# Build SEC EDGAR Parquet Archive

Reads all TSV files from the SEC EDGAR Financial Statement Data Set bulk downloads,
concatenates like-tables across all 78 period folders, and writes partitioned Parquet
to `sec-data-pqt/`.

**Do not re-run casually** — `num.tsv` is ~835 MB per folder × 78 folders ≈ 65 GB of raw reads.
The streaming engine means memory stays bounded, but wall time is significant.

## Source data layout

- Quarterly format (2009–2024): `2009q1_notes/`, `2009q2_notes/`, …, `2024q4_notes/`
- Monthly format (2025+): `2025_01_notes/`, `2025_02_notes/`, …, `2026_02_notes/`

Each folder contains: `sub.tsv`, `num.tsv`, `txt.tsv`, `tag.tsv`, `dim.tsv`,
`pre.tsv`, `ren.tsv`, `cal.tsv`

## Source-folder columns added to every row

| Column | Example | Meaning |
|---|---|---|
| `src_folder` | `"2009q1_notes"` | Exact folder name — unambiguous provenance |
| `src_year`   | `2009` | Calendar year of the bulk release |
| `src_period` | `"2009q1"` or `"2025_01"` | Quarter or month identifier stripped of `_notes` |

These are **data-availability dates** (when the bulk drop was published),
distinct from `period` / `ddate` columns already in the data (which describe
what the filing covers).

## Hive-partitioning strategy

| Table | Strategy | Rationale |
|---|---|---|
| `num`, `txt` | Partition by `src_year` | These are the largest tables (num alone is ~65 GB raw across all years). Partitioning by year lets downstream `scan_parquet` calls push `src_year` predicates and skip entire year directories. |
| `sub`, `tag`, `dim`, `ren`, `pre`, `cal` | Single parquet file | These tables are small enough that a single file is simpler to read and the scan overhead of partitioning outweighs the benefit. |

Output tree:
```
sec-data-pqt/
  num/src_year=2009/00000000.parquet
  num/src_year=2010/00000000.parquet
  ...
  txt/src_year=2009/00000000.parquet
  ...
  sub/sub.parquet
  tag/tag.parquet
  dim/dim.parquet
  ren/ren.parquet
  pre/pre.parquet
  cal/cal.parquet
```

Re-reading partitioned tables later:
```python
pl.scan_parquet("sec-data-pqt/num/**/*.parquet", hive_partitioning=True)
    .filter(pl.col("src_year") >= 2020)
```

## Schema-drift handling

Early quarterly files (2009–early 2010s) were published before the SEC standardized
its XBRL toolchain. `scan_csv` on those folders infers most columns as `String`
where modern files are `Int64` or `Float64`.

Strategy: supply explicit `schema_overrides` for the columns that drift, casting
them to the modern type. Using `ignore_errors=False` (the default) means a
genuine parse failure will raise loudly rather than silently coercing to null.

For `num.value`, `Float64` is correct and safe across all eras — the column
holds XBRL numeric facts and values like `0` parse fine as Float64.

Two additional parsing fixes are required across the full archive:

- **`quote_char=None`** on every table: SEC TSVs are unquoted but some fields (`tag.doc`, `txt.value`) contain embedded newlines that confuse Polars' default quoting logic, causing a "CSV malformed" error in small files.
- **`truncate_ragged_lines=True`** for `dim` and `txt` only: `dim.segments` contains embedded tabs in modern files; a handful of early `txt` folders have extra tab-split fields beyond the 20-column schema. Both are safely clipped by this flag with no meaningful data loss.

In [1]:
import re
from pathlib import Path

import polars as pl

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_DIR = Path("/home/sulli/research/tra_automation/fin_stmt_and_notes_data")
OUT_DIR  = Path("/home/sulli/research/tra_automation/sec-data-pqt")
OUT_DIR.mkdir(exist_ok=True)

# ── Schema overrides ─────────────────────────────────────────────────────────
# Applied on every scan_csv call to enforce a consistent dtype regardless of
# what the per-folder inference produces.
#
# Rule: use the dtype from modern monthly files; cast older files to match.
# Columns not listed here are left to inference (String is safe for join keys).
#
# Date columns (YYYYMMDD) are kept as String throughout — consistent with
# `period` and `filed` — to avoid opaque integer arithmetic on dates.

SCHEMA_OVERRIDES = {
    "num": {
        "ddate":    pl.String,   # YYYYMMDD — keep as string; cast to Date downstream if needed
        "qtrs":     pl.Int64,
        "iprx":     pl.Int64,
        "value":    pl.Float64,
        "footlen":  pl.Int64,
        "dimn":     pl.Int64,
        "durp":     pl.Float64,
        "datp":     pl.Float64,
        "dcml":     pl.Int64,
    },
    "txt": {
        "ddate":    pl.String,
        "qtrs":     pl.Int64,
        "iprx":     pl.Int64,
        "dcml":     pl.Int64,
        "durp":     pl.Float64,
        "datp":     pl.Float64,
        "dimn":     pl.Int64,
        "escaped":  pl.Int64,
        "srclen":   pl.Int64,
        "txtlen":   pl.Int64,
        "footlen":  pl.Int64,
    },
    "sub": {
        "cik":         pl.Int64,
        "sic":         pl.Int64,
        "ein":         pl.Int64,
        "changed":     pl.String,  # YYYYMMDD date — keep as string
        "wksi":        pl.Int64,
        "fye":         pl.Int64,
        "period":      pl.String,  # YYYYMMDD date — keep as string
        "fy":          pl.Int64,
        "filed":       pl.String,  # YYYYMMDD date — keep as string
        "prevrpt":     pl.Int64,
        "detail":      pl.Int64,
        "nciks":       pl.Int64,
        "aciks":         pl.String,
        "pubfloatusd": pl.Float64,
        "floatdate":   pl.String,
        "floatmems":   pl.Int64,
    },
    "tag": {
        "custom":   pl.Int64,
        "abstract": pl.Int64,
    },
    "dim": {
        "segt": pl.String,   # early files: "0"; modern files: free-form text (e.g. company names)
    },
    "pre": {
        "report":   pl.Int64,
        "line":     pl.Int64,
        "inpth":    pl.Int64,
        "negating": pl.Int64,
    },
    "ren": {
        "report":       pl.Int64,
        "parentreport": pl.Int64,
        "ultparentrpt": pl.Int64,
    },
    "cal": {
        "grp":      pl.Int64,
        "arc":      pl.Int64,
        "negative": pl.Int64,
    },
}

print(f"Output directory: {OUT_DIR}")
print(f"Schema overrides defined for tables: {list(SCHEMA_OVERRIDES)}")

Output directory: /home/sulli/research/tra_automation/sec-data-pqt
Schema overrides defined for tables: ['num', 'txt', 'sub', 'tag', 'dim', 'pre', 'ren', 'cal']


## Discover and parse period folders

In [2]:
# ── Folder discovery ─────────────────────────────────────────────────────────
# Pattern A (quarterly): 2009q1_notes  → year=2009, period="2009q1"
# Pattern B (monthly):   2025_01_notes → year=2025, period="2025_01"
#
# Monthly regex restricts to valid months (01–12) so a typo like 2025_00_notes
# or a new SEC format variant raises ValueError via _parse_folder rather than
# being silently ingested with a nonsensical src_period.

_QUARTERLY_RE = re.compile(r"^(\d{4})(q\d)_notes$", re.IGNORECASE)
_MONTHLY_RE   = re.compile(r"^(\d{4})_(0[1-9]|1[0-2])_notes$")


def _parse_folder(name: str) -> tuple[str, int, str]:
    """Return (folder_name, src_year, src_period) for a notes folder name."""
    m = _QUARTERLY_RE.match(name)
    if m:
        year, quarter = int(m.group(1)), m.group(2).lower()
        return name, year, f"{year}{quarter}"
    m = _MONTHLY_RE.match(name)
    if m:
        year, month = int(m.group(1)), m.group(2)
        return name, year, f"{year}_{month}"
    raise ValueError(f"Unrecognised folder name format: {name!r}")


folders: list[tuple[str, int, str]] = sorted(
    [_parse_folder(p.name) for p in DATA_DIR.iterdir() if p.is_dir()],
    key=lambda t: (t[1], t[2]),  # sort by year then period string
)

print(f"Found {len(folders)} period folders")
print("First 5:", folders[:5])
print("Last  5:", folders[-5:])


Found 78 period folders
First 5: [('2009q1_notes', 2009, '2009q1'), ('2009q2_notes', 2009, '2009q2'), ('2009q3_notes', 2009, '2009q3'), ('2009q4_notes', 2009, '2009q4'), ('2010q1_notes', 2010, '2010q1')]
Last  5: [('2025_10_notes', 2025, '2025_10'), ('2025_11_notes', 2025, '2025_11'), ('2025_12_notes', 2025, '2025_12'), ('2026_01_notes', 2026, '2026_01'), ('2026_02_notes', 2026, '2026_02')]


## Helper: build union LazyFrame for one table

In [3]:
def build_lazy(table: str) -> pl.LazyFrame:
    """
    Return a LazyFrame that is the union of `table`.tsv across all period
    folders, with three source-provenance columns prepended:
        src_folder  String  e.g. "2009q1_notes"
        src_year    Int32   e.g. 2009
        src_period  String  e.g. "2009q1" or "2025_01"

    Schema overrides in SCHEMA_OVERRIDES[table] are applied on every scan so
    that dtype mismatches across era don't prevent concatenation.
    """
    overrides = SCHEMA_OVERRIDES.get(table, {})
    # dim.tsv has embedded tabs in the segments field; early txt.tsv has extra tab-split fields
    truncate_ragged = table in ("dim", "txt")
    frames: list[pl.LazyFrame] = []

    for folder_name, src_year, src_period in folders:
        tsv_path = DATA_DIR / folder_name / f"{table}.tsv"
        if not tsv_path.exists():
            # A handful of early quarterly drops may be missing some tables.
            # Skip silently — the missing folder is still identifiable via the
            # gap in src_period values when you read the parquet later.
            print(f"  [SKIP] {tsv_path} not found")
            continue

        lf = (
            pl.scan_csv(
                tsv_path,
                separator="\t",
                infer_schema_length=500,
                schema_overrides=overrides if overrides else None,
                null_values=["", "NULL", "null", "None"],
                quote_char=None,              # SEC TSVs are unquoted; None prevents misparse of embedded newlines in doc fields
                truncate_ragged_lines=truncate_ragged,  # dim.tsv has embedded tabs in segments field
                ignore_errors=False,          # fail loudly on genuine parse errors
            )
            .with_columns(
                pl.lit(folder_name).alias("src_folder"),
                pl.lit(src_year).cast(pl.Int32).alias("src_year"),
                pl.lit(src_period).alias("src_period"),
            )
        )
        frames.append(lf)

    if not frames:
        raise RuntimeError(f"No TSV files found for table '{table}'")

    # diagonal_relaxed allows concat even if a rare column appears in only some
    # folders (fills missing columns with null rather than raising).
    return pl.concat(frames, how="diagonal_relaxed")


print("build_lazy() helper defined.")

build_lazy() helper defined.


## Write small tables (single parquet files)

`sub`, `tag`, `dim`, `ren`, `pre`, and `cal` are written as single parquet files per table.
No hive partitioning — simpler to `scan_parquet("sec-data-pqt/pre/pre.parquet")` and the
scan overhead of partitioning would outweigh any benefit for these sizes.

Each output file is deleted before writing. `sink_parquet` appends row groups
to an existing file rather than overwriting, so without this step a re-run
would silently duplicate all rows.

In [4]:
for tbl in ["sub", "tag", "dim", "ren", "pre", "cal"]:
    out_dir = OUT_DIR / tbl
    out_dir.mkdir(exist_ok=True)
    out_path = out_dir / f"{tbl}.parquet"
    out_path.unlink(missing_ok=True)  # prevent row-group appending on re-run
    print(f"Writing {tbl} \u2192 {out_path} ...")
    build_lazy(tbl).sink_parquet(
        out_path,
        compression="zstd",
        mkdir=True,
    )
    print(f"  done: {out_path}")


Writing sub → /home/sulli/research/tra_automation/sec-data-pqt/sub/sub.parquet ...


  done: /home/sulli/research/tra_automation/sec-data-pqt/sub/sub.parquet
Writing tag → /home/sulli/research/tra_automation/sec-data-pqt/tag/tag.parquet ...


  done: /home/sulli/research/tra_automation/sec-data-pqt/tag/tag.parquet
Writing dim → /home/sulli/research/tra_automation/sec-data-pqt/dim/dim.parquet ...


  done: /home/sulli/research/tra_automation/sec-data-pqt/dim/dim.parquet
Writing ren → /home/sulli/research/tra_automation/sec-data-pqt/ren/ren.parquet ...


  done: /home/sulli/research/tra_automation/sec-data-pqt/ren/ren.parquet
Writing pre → /home/sulli/research/tra_automation/sec-data-pqt/pre/pre.parquet ...


  done: /home/sulli/research/tra_automation/sec-data-pqt/pre/pre.parquet
Writing cal → /home/sulli/research/tra_automation/sec-data-pqt/cal/cal.parquet ...


  done: /home/sulli/research/tra_automation/sec-data-pqt/cal/cal.parquet


## Write large tables with hive partitioning by `src_year`

`num` and `txt` are partitioned by `src_year`. This produces
one subdirectory per year (e.g., `num/src_year=2009/`) so downstream queries
can filter on `src_year` and skip entire year directories.

The `pl.PartitionBy` API is marked unstable in Polars 1.38 — it may change in
future versions. If you upgrade Polars and this breaks, check `help(pl.PartitionBy)`.

`PartitionBy` writes a fixed filename (`00000000.parquet`) inside each partition
directory and overwrites it on re-run, so no pre-deletion is needed here.

In [5]:
for tbl in ["num", "txt"]:
    out_dir = OUT_DIR / tbl
    out_dir.mkdir(exist_ok=True)
    print(f"Writing {tbl} \u2192 {out_dir}/src_year=*/ ...")
    build_lazy(tbl).sink_parquet(
        pl.PartitionBy(
            out_dir,
            key="src_year",
            include_key=True,   # keep src_year IN the parquet files too
        ),
        compression="zstd",
        mkdir=True,
    )
    print(f"  done: {out_dir}")


Writing num → /home/sulli/research/tra_automation/sec-data-pqt/num/src_year=*/ ...


  done: /home/sulli/research/tra_automation/sec-data-pqt/num
Writing txt → /home/sulli/research/tra_automation/sec-data-pqt/txt/src_year=*/ ...


  done: /home/sulli/research/tra_automation/sec-data-pqt/txt


## Verification spot-check

Quick sanity check: confirm each output exists, print row count for `sub`
(small, fast), and show schema for `num` (tests that dtype coercion worked).

In [6]:
# Small tables — direct path
for tbl in ["sub", "tag", "dim", "ren", "pre", "cal"]:
    path = OUT_DIR / tbl / f"{tbl}.parquet"
    lf = pl.scan_parquet(path)
    n = lf.select(pl.len()).collect().item()
    print(f"{tbl}: {n:,} rows  |  {lf.collect_schema()}")

# Large tables — hive scan
for tbl in ["num", "txt"]:
    path = OUT_DIR / tbl / "**" / "*.parquet"
    lf = pl.scan_parquet(str(path), hive_partitioning=True)
    n = lf.select(pl.len()).collect().item()
    print(f"{tbl}: {n:,} rows  |  {lf.collect_schema()}")


sub: 823,056 rows  |  Schema({'adsh': String, 'cik': Int64, 'name': String, 'sic': Int64, 'countryba': String, 'stprba': String, 'cityba': String, 'zipba': String, 'bas1': String, 'bas2': String, 'baph': String, 'countryma': String, 'stprma': String, 'cityma': String, 'zipma': String, 'mas1': String, 'mas2': String, 'countryinc': String, 'stprinc': String, 'ein': Int64, 'former': String, 'changed': String, 'afs': String, 'wksi': Int64, 'fye': Int64, 'form': String, 'period': String, 'fy': Int64, 'fp': String, 'filed': String, 'accepted': String, 'prevrpt': Int64, 'detail': Int64, 'instance': String, 'nciks': Int64, 'aciks': String, 'pubfloatusd': Float64, 'floatdate': String, 'floataxis': String, 'floatmems': Int64, 'src_folder': String, 'src_year': Int32, 'src_period': String})
tag: 44,488,996 rows  |  Schema({'tag': String, 'version': String, 'custom': Int64, 'abstract': Int64, 'datatype': String, 'iord': String, 'crdr': String, 'tlabel': String, 'doc': String, 'src_folder': String, 

cal: 37,188,738 rows  |  Schema({'adsh': String, 'grp': Int64, 'arc': Int64, 'negative': Int64, 'ptag': String, 'pversion': String, 'ctag': String, 'cversion': String, 'src_folder': String, 'src_year': Int32, 'src_period': String})


num: 389,603,167 rows  |  Schema({'adsh': String, 'tag': String, 'version': String, 'ddate': String, 'qtrs': Int64, 'uom': String, 'dimh': String, 'iprx': Int64, 'value': Float64, 'footnote': String, 'footlen': Int64, 'dimn': Int64, 'coreg': String, 'durp': Float64, 'datp': Float64, 'dcml': Int64, 'src_folder': String, 'src_year': Int32, 'src_period': String})
txt: 48,381,860 rows  |  Schema({'adsh': String, 'tag': String, 'version': String, 'ddate': String, 'qtrs': Int64, 'iprx': Int64, 'lang': String, 'dcml': Int64, 'durp': Float64, 'datp': Float64, 'dimh': String, 'dimn': Int64, 'coreg': String, 'escaped': Int64, 'srclen': Int64, 'txtlen': Int64, 'footnote': String, 'footlen': Int64, 'context': String, 'value': String, 'src_folder': String, 'src_year': Int32, 'src_period': String})
